In [1]:
%matplotlib inline

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import random
import ipywidgets as widgets
from IPython.display import display
plt.style.use('seaborn-v0_8-whitegrid')

In [3]:
class Paciente:
    def __init__(self, teta_real, d_real, m_real, P0, sigma):
        self.teta_real = np.array(teta_real)
        self.d = int(d_real)
        self.m = int(m_real)
        self.P0 = P0
        self.sigma = sigma
        self.P_histo = np.zeros(200)
        self.I_histo = np.zeros(200)

    def step(self, I_atual_k_menos_1):
        self.I_histo = np.roll(self.I_histo, 1)
        self.I_histo[0] = I_atual_k_menos_1
        P_ante_1 = self.P_histo[0]
        idx_d = self.d - 1
        I_ante_d  = self.I_histo[idx_d]  if idx_d  < len(self.I_histo) else 0.0
        idx_dm    = self.d + self.m - 1
        I_ante_dm = self.I_histo[idx_dm] if idx_dm < len(self.I_histo) else 0.0
        fi = np.array([-P_ante_1, I_ante_d, I_ante_dm])
        ruido_e = random.gauss(0, self.sigma)
        P_k = self.teta_real @ fi + ruido_e
        MAP = self.P0 - P_k
        self.P_histo = np.roll(self.P_histo, 1)
        self.P_histo[0] = P_k
        return MAP, P_k, ruido_e

In [4]:
class Identificador:
    def __init__(self, lambda_val, d_vetor, m_vetor, N_sim):
        self.lambda_val  = lambda_val
        self.d_vetor     = np.array(d_vetor)
        self.m_vetor     = np.array(m_vetor)
        self.k           = 0
        self.teta_ea     = np.zeros(12)
        self.Pant        = np.kron(np.eye(4), 10 * np.eye(3))
        self.histo_erro  = np.zeros((N_sim, 4))
        self.fator       = np.zeros(4)
        self.P_histo     = np.zeros(100)
        self.I_histo     = np.zeros(100)
        self.ind_melhor  = 0
        self.teta_melhor = np.zeros(3)

    def RLS(self, P_k_medido, I_k_anterior):
        self.k += 1
        self.P_histo = np.roll(self.P_histo, 1);  self.P_histo[0] = P_k_medido
        self.I_histo = np.roll(self.I_histo, 1);  self.I_histo[0] = I_k_anterior

        fia = np.zeros(12)
        for j in range(4):
            ind  = 3 * j
            d_j, m_j = self.d_vetor[j], self.m_vetor[j]
            fia[ind] = -self.P_histo[1]
            if d_j - 1       < len(self.I_histo): fia[ind + 1] = self.I_histo[d_j - 1]
            if d_j + m_j - 1 < len(self.I_histo): fia[ind + 2] = self.I_histo[d_j + m_j - 1]

        Soma_mi = 0
        for j in range(4):
            sl    = slice(3*j, 3*j+3)
            fia_j = fia[sl];  Pant_j = self.Pant[sl, sl]
            den   = self.lambda_val + fia_j @ Pant_j @ fia_j
            if den == 0: den = 1e-9
            kk_j  = (Pant_j @ fia_j) / den
            teta_ant = self.teta_ea[sl]
            self.teta_ea[sl] = teta_ant + kk_j * (P_k_medido - teta_ant @ fia_j)
            self.Pant[sl, sl] = (np.eye(3) - np.outer(kk_j, fia_j)) @ Pant_j / self.lambda_val
            ep = (P_k_medido - self.teta_ea[sl] @ fia_j)**2
            self.histo_erro[self.k-1, j] = ep
            janela = 20
            ini = max(0, self.k - janela)
            self.fator[j] = np.sum(self.histo_erro[ini:self.k, j])
            Soma_mi += 1.0 / (self.fator[j] + 1e-9)

        Med_adeq = np.array([1.0/(self.fator[j]+1e-9) for j in range(4)]) / (Soma_mi + 1e-9)
        ind_cand = np.argmax(Med_adeq)
        if Med_adeq[ind_cand] > 1.35 * Med_adeq[self.ind_melhor]:
            self.ind_melhor = ind_cand

        sl_best = slice(3*self.ind_melhor, 3*self.ind_melhor+3)
        self.teta_melhor = self.teta_ea[sl_best]
        incertezas = np.diagonal(self.Pant)[[1, 4, 7, 10]]
        P_melhor = self.Pant[sl_best, sl_best].copy()
        return self.teta_melhor, self.d_vetor[self.ind_melhor], self.m_vetor[self.ind_melhor], incertezas, self.fator.copy(), P_melhor


In [ ]:



class Controlador_GPC_Adaptativo:
    def __init__(self, ro=1.5):
        self.ro = ro
        self.k  = 0
        self.I_ante_1 = 0.0
        self.I_histo  = np.zeros(100)

    def calcular_controle(self, P_ref, P_k_medido, teta_est, d_est, m_est, inovacao_k, P_cov):
        # P_cov ignorado — usa RLS mas sem dualidade via covariância
        self.k += 1
        a1_est, b1_est, bm_est = teta_est
        b1_ctrl = max(b1_est, 0.01)

        if self.k <= 10:
            PRBS = 10.0
            if self.k > 5:
                PRBS = 60 - (60 / 3.8) * abs(b1_est)
            I_calculado = PRBS if random.random() > 0.5 else 0.0
            I_calculado = np.clip(I_calculado, 0, 180)
            self.I_histo = np.roll(self.I_histo, 1)
            self.I_histo[0] = I_calculado
            self.I_ante_1 = I_calculado
            return I_calculado

        k0 = (-a1_est) ** d_est * P_k_medido
        for i in range(1, d_est):
            t1 = b1_ctrl * self.I_histo[i]         if i         < len(self.I_histo) else 0.0
            t2 = bm_est  * self.I_histo[m_est + i] if m_est + i < len(self.I_histo) else 0.0
            k0 += (-a1_est) ** i * (t1 + t2)
        if m_est < len(self.I_histo):
            k0 += bm_est * self.I_histo[m_est]

        num = b1_ctrl * (P_ref - k0) + self.ro * self.I_ante_1
        den = b1_ctrl ** 2 + self.ro
        I_calculado = np.clip(num / max(den, 1e-6), 0, 180)

        self.I_histo = np.roll(self.I_histo, 1)
        self.I_histo[0] = I_calculado
        self.I_ante_1 = I_calculado
        return I_calculado

class Controlador_32_Perturbacao:
    """
    Seção 3.2 de Jamier: u_perturb(t) = u_ca(t) + u_p(t)
    A base é o controlador CAUTELOSO (Eq. 2.23), não o GPC sem ro.
    O sinal de perturbação u_p é somado para garantir excitação persistente.
    """
    def __init__(self, amplitude=8.0, taxa_decaimento=0.997, seed=None):
        self.k        = 0
        self.I_ante_1 = 0.0
        self.I_histo  = np.zeros(100)
        self._amp     = amplitude
        self._decay   = taxa_decaimento
        self._rng     = np.random.default_rng(seed)

    def calcular_controle(self, P_ref, P_k_medido, teta_est, d_est, m_est, inovacao_k, P_cov):
        self.k += 1
        a1_est, b1_est, bm_est = teta_est
        b1_ctrl = max(b1_est, 0.01)

        if self.k <= 10:
            I_calculado = 30.0 if self._rng.integers(0, 2) == 1 else 0.0
            I_calculado = np.clip(I_calculado, 0, 180)
            self.I_histo = np.roll(self.I_histo, 1);  self.I_histo[0] = I_calculado
            self.I_ante_1 = I_calculado
            return I_calculado

        # K0 padrão
        k0 = (-a1_est) ** d_est * P_k_medido
        for i in range(1, d_est):
            t1 = b1_ctrl * self.I_histo[i]         if i         < len(self.I_histo) else 0.0
            t2 = bm_est  * self.I_histo[m_est + i] if m_est + i < len(self.I_histo) else 0.0
            k0 += (-a1_est) ** i * (t1 + t2)
        if m_est < len(self.I_histo):
            k0 += bm_est * self.I_histo[m_est]

        # --- Controlador cauteloso (Eq. 2.23) como base ---
        # p_b1: variância de b1 da matriz de covariância
        # Cauteloso (Eq. 2.23) — base do 3.2
        p_b1 = P_cov[1, 1]  # limita para não zerar o controle no início
        IP_teta = P_cov[1, :] @ teta_est

        num_ca = b1_ctrl * (P_ref - k0) + IP_teta + p_b1 * self.I_ante_1
        den_ca = b1_ctrl ** 2 + p_b1
        u_ca   = num_ca / max(den_ca, 1e-6)

        # --- Perturbação PRBS — dualidade explícita (Seção 3.2) ---
        u_p       = self._amp * (self._rng.integers(0, 2) * 2 - 1)
        self._amp *= self._decay

        I_calculado = np.clip(u_ca + u_p, 0, 180)
        self.I_histo = np.roll(self.I_histo, 1);  self.I_histo[0] = I_calculado
        self.I_ante_1 = I_calculado
        return I_calculado


class Controlador_33_Restricoes:
    """
    Seção 3.3 de Jamier — Jacobs e Hughes (1974): Eq. 3.2.
    Base: controlador cauteloso (Eq. 2.23).
    Restrição: impede que |u_ca| caia abaixo de u_lim,
    evitando o fenômeno de turn-off.
    """
    def __init__(self, u_lim=5.0):
        self.u_lim    = u_lim   # limite mínimo de excitação
        self.k        = 0
        self.I_ante_1 = 0.0
        self.I_histo  = np.zeros(100)

    def calcular_controle(self, P_ref, P_k_medido, teta_est, d_est, m_est, inovacao_k, P_cov):
        self.k += 1
        a1_est, b1_est, bm_est = teta_est
        b1_ctrl = max(b1_est, 0.01)

        if self.k <= 10:
            PRBS = 10.0
            if self.k > 5:
                PRBS = 60 - (60 / 3.8) * abs(b1_est)
            I_calculado = PRBS if random.random() > 0.5 else 0.0
            I_calculado = np.clip(I_calculado, 0, 180)
            self.I_histo = np.roll(self.I_histo, 1);  self.I_histo[0] = I_calculado
            self.I_ante_1 = I_calculado
            return I_calculado

        k0 = (-a1_est) ** d_est * P_k_medido
        for i in range(1, d_est):
            t1 = b1_ctrl * self.I_histo[i]         if i         < len(self.I_histo) else 0.0
            t2 = bm_est  * self.I_histo[m_est + i] if m_est + i < len(self.I_histo) else 0.0
            k0 += (-a1_est) ** i * (t1 + t2)
        if m_est < len(self.I_histo):
            k0 += bm_est * self.I_histo[m_est]

        # Base: cauteloso real (Eq. 2.23)
        p_b1    = P_cov[1, 1]
        IP_teta = P_cov[1, :] @ teta_est

        num_ca = b1_ctrl * (P_ref - k0) + IP_teta + p_b1 * self.I_ante_1
        den_ca = b1_ctrl ** 2 + p_b1
        u_ca   = num_ca / max(den_ca, 1e-6)

        # Restrição de Jacobs e Hughes (Eq. 3.2):
        # se o cauteloso for fraco demais, força u_lim para manter excitação
        if abs(u_ca) < self.u_lim:
            I_calculado = self.u_lim * np.sign(u_ca) if u_ca != 0 else self.u_lim
        else:
            I_calculado = u_ca

        I_calculado = np.clip(I_calculado, 0, 180)
        self.I_histo = np.roll(self.I_histo, 1);  self.I_histo[0] = I_calculado
        self.I_ante_1 = I_calculado
        return I_calculado


class Controlador_35_Inovacoes:
    """
    Seção 3.5 de Jamier — IDC (Milito e Cadorin, 1982): Eq. 3.13.
    Compromisso entre controlador cauteloso (λ=1) e equivalente à certeza (λ=0).
    Usa P_cov diretamente — sem suavização, sem energia intermediária.
    """
    def __init__(self, lambda_idc=0.5):
        self.lambda_idc = lambda_idc
        self.k          = 0
        self.I_ante_1   = 0.0
        self.I_histo    = np.zeros(100)

    def calcular_controle(self, P_ref, P_k_medido, teta_est, d_est, m_est, inovacao_k, P_cov):
        self.k += 1
        a1_est, b1_est, bm_est = teta_est
        b1_ctrl = max(b1_est, 0.01)

        if self.k <= 10:
            PRBS = 10.0
            if self.k > 5:
                PRBS = 60 - (60 / 3.8) * abs(b1_est)
            I_calculado = PRBS if random.random() > 0.5 else 0.0
            I_calculado = np.clip(I_calculado, 0, 180)
            self.I_histo = np.roll(self.I_histo, 1);  self.I_histo[0] = I_calculado
            self.I_ante_1 = I_calculado
            return I_calculado

        k0 = (-a1_est) ** d_est * P_k_medido
        for i in range(1, d_est):
            t1 = b1_ctrl * self.I_histo[i]         if i         < len(self.I_histo) else 0.0
            t2 = bm_est  * self.I_histo[m_est + i] if m_est + i < len(self.I_histo) else 0.0
            k0 += (-a1_est) ** i * (t1 + t2)
        if m_est < len(self.I_histo):
            k0 += bm_est * self.I_histo[m_est]

        # Eq. 3.13 de Jamier — IDC
        lam     = self.lambda_idc
        p_b1 = P_cov[1, 1]  # limita para não zerar o controle no início
        IP_teta = P_cov[1, :] @ teta_est

        num = b1_ctrl * (P_ref - k0) + (1 - lam) * IP_teta + (1 - lam) * p_b1 * self.I_ante_1
        den = b1_ctrl ** 2 + (1 - lam) * p_b1

        I_calculado = np.clip(num / max(den, 1e-6), 0, 180)

        self.I_histo = np.roll(self.I_histo, 1);  self.I_histo[0] = I_calculado
        self.I_ante_1 = I_calculado
        return I_calculado


In [6]:
#COMPARATIVO

def run_simulation(ro, lambda_val, P0, map_ref, sigma,
                   a1_ini, b1_ini, bm_ini, d_ini, m_ini,
                   k_change, a1_new, b1_new, bm_new, d_new, m_new,
                   N_MC=50):

    N     = 500
    yref  = P0 - map_ref
    d_vetor = [2, 3, 4, 5]
    m_vetor = [2, 3, 4, 5]
    teta_real_ini = [a1_ini, b1_ini, bm_ini]
    t    = np.arange(N)
    pref = np.full(N, map_ref)

    # Configuração dos 5 controladores
    # make_ctrl é uma lambda para recriar o objeto a cada run MC
    cfg = [
    ('GPC Fixo',        'seagreen',
     lambda: Controlador_GPC_Fixo([a1_ini, b1_ini, bm_ini], d_ini, m_ini, ro)),
    ('GPC Adaptativo',  'mediumpurple',
     lambda: Controlador_GPC_Adaptativo(ro)),
    ('3.2 Perturbação', 'royalblue',
     lambda: Controlador_32_Perturbacao(amplitude=8.0, taxa_decaimento=0.997, seed=None)),
    ('3.3 Restrições',  'crimson',
     lambda: Controlador_33_Restricoes(u_lim=15.0)),
    ('3.5 IDC',         'darkorange',
     lambda: Controlador_35_Inovacoes(lambda_idc=0.5)),   # <- aqui estava alpha=3.0, beta=0.9
    ]

    #  Acumuladores
    res = {}
    for nome, cor, _ in cfg:
        res[nome] = dict(
            cor          = cor,
            all_map      = np.zeros((N_MC, N)),
            all_i        = np.zeros((N_MC, N)),
            # guardamos a última run para gráficos não-estocásticos
            delay_h      = np.zeros(N),
            delay_real   = np.zeros(N),
            teta_all     = np.zeros((N, 12)),
            teta_real    = np.zeros((N, 3)),
            custos       = np.zeros((N, 4)),
            incerteza    = np.zeros((N, 4)),
        )

    # Loop Monte Carlo
    for nome, _, make_ctrl in cfg:
        r = res[nome]
        for mc in range(N_MC):
            paciente      = Paciente(teta_real_ini, d_ini, m_ini, P0, sigma)
            identificador = Identificador(lambda_val, d_vetor, m_vetor, N)
            ctrl          = make_ctrl()

            map_h  = np.zeros(N);  i_h   = np.zeros(N)
            d_h    = np.zeros(N);  dr_h  = np.zeros(N)
            ta_h   = np.zeros((N, 12));  tr_h = np.zeros((N, 3))
            cu_h   = np.zeros((N, 4));   inc_h = np.zeros((N, 4))

            I_pac = 0.0

            # DEPOIS — rampa de 50 passos
            RAMPA = 50  # passos para a transição

            for k in range(N):
                # Rampa gradual dos parâmetros
                if k_change <= k < k_change + RAMPA:
                    alpha = (k - k_change) / RAMPA
                    paciente.teta_real = np.array([
                        a1_ini + alpha * (a1_new - a1_ini),
                        b1_ini + alpha * (b1_new - b1_ini),
                        bm_ini + alpha * (bm_new - bm_ini),
                    ])

                # Mudança completa só ao fim da rampa
                elif k == k_change + RAMPA:
                    paciente.teta_real = np.array([a1_new, b1_new, bm_new])
                    paciente.d = int(d_new)
                    paciente.m = int(m_new)
                    identificador.Pant = np.kron(np.eye(4), 10 * np.eye(3))

                tr_h[k] = paciente.teta_real;  dr_h[k] = paciente.d

                MAP, P_med, _ = paciente.step(I_pac)
                teta_est, d_est, m_est, vetor_inc, custos_step, P_melhor = identificador.RLS(P_med, I_pac)
                var_b1  = vetor_inc[identificador.ind_melhor]
                I_atual = ctrl.calcular_controle(yref, P_med, teta_est, d_est, m_est, var_b1, P_melhor)

                I_pac    = I_atual
                map_h[k] = MAP;     i_h[k]  = I_atual
                d_h[k]   = d_est;   ta_h[k] = identificador.teta_ea
                cu_h[k]  = custos_step;  inc_h[k] = vetor_inc

            r['all_map'][mc] = map_h;  r['all_i'][mc] = i_h
            # sobrescreve — guarda a última run
            r['delay_h']   = d_h;   r['delay_real'] = dr_h
            r['teta_all']  = ta_h;  r['teta_real']  = tr_h
            r['custos']    = cu_h;  r['incerteza']  = inc_h

    # Funções auxiliares de plot
    def _vline(ax):
        ax.axvline(x=k_change, color='black', linestyle=':', linewidth=1.2,
                   label='Mudança paciente')

    def _envelope(ax, nome, arr_mc):
        cor   = res[nome]['cor']
        media = arr_mc.mean(axis=0)
        ax.plot(t, media, color=cor, linewidth=2, label=nome)

    # FIGURA 1 — MAP, Infusão, Atraso, a1, b1, bm

    fig, axes = plt.subplots(3, 2, figsize=(15, 12))
    fig.suptitle(f'Comparativo Monte Carlo — 5 Controladores  (N_MC={N_MC})',
                 fontsize=14, fontweight='bold')

    # MAP
    ax = axes[0, 0]
    for nome, _, _ in cfg:
        _envelope(ax, nome, res[nome]['all_map'])
    ax.plot(t, pref, 'r--', linewidth=1.5, label=f'Ref ({map_ref} mmHg)')
    _vline(ax)
    ax.set_ylim(50, 200);  ax.set_title('Pressão Arterial — mmHg')
    ax.legend(fontsize=8, loc='upper right');  ax.grid(True)

    # Infusão
    ax = axes[1, 0]
    for nome, _, _ in cfg:
        _envelope(ax, nome, res[nome]['all_i'])
    _vline(ax)
    ax.set_ylim(-10, 190);  ax.set_title('Taxa de Infusão de SNP — ml/h')
    ax.legend(fontsize=8, loc='upper right');  ax.grid(True)

    # Parâmetro a1
    ax = axes[0, 1]
    nome_ref = 'GPC Fixo'  # referência explícita para o valor real
    for nome, _, _ in cfg:
        ax.plot(t, res[nome]['teta_all'][:, 0], color=res[nome]['cor'],
                linewidth=1.2, alpha=0.85, label=nome)
    ax.plot(t, res[nome_ref]['teta_real'][:, 0], 'k--', linewidth=2, label='a1 Real')
    _vline(ax)
    ax.set_ylim(-1.5, 0.5);  ax.set_title('Parâmetro a1')
    ax.legend(fontsize=8);  ax.grid(True)

    # Parâmetro b1
    ax = axes[1, 1]
    for nome, _, _ in cfg:
        ax.plot(t, res[nome]['teta_all'][:, 1], color=res[nome]['cor'],
                linewidth=1.2, alpha=0.85, label=nome)
    ax.plot(t, res[nome_ref]['teta_real'][:, 1], 'k--', linewidth=2, label='b1 Real')
    _vline(ax)
    ax.set_ylim(-0.5, 1.5);  ax.set_title('Parâmetro b1')
    ax.legend(fontsize=8);  ax.grid(True)

    # Parâmetro bm
    ax = axes[2, 1]
    for nome, _, _ in cfg:
        ax.plot(t, res[nome]['teta_all'][:, 2], color=res[nome]['cor'],
                linewidth=1.2, alpha=0.85, label=nome)
    ax.plot(t, res[nome_ref]['teta_real'][:, 2], 'k--', linewidth=2, label='bm Real')
    _vline(ax)
    ax.set_ylim(-1.0, 1.5);  ax.set_title('Parâmetro bm')
    ax.legend(fontsize=8);  ax.grid(True)

    # Atraso
    ax = axes[2, 0]
    for nome, _, _ in cfg:
        ax.plot(t, res[nome]['delay_h'], color=res[nome]['cor'],
                linewidth=1.5, label=nome)
    ax.plot(t, res[nome_ref]['delay_real'], 'k--', linewidth=2, label='Atraso Real')
    _vline(ax)
    ax.set_ylim(1, 6);  ax.set_title('Atraso Estimado vs Real')
    ax.legend(fontsize=8);  ax.grid(True)

    plt.tight_layout()
    plt.show()

    # FIGURA 2 — Custo Acumulado (1 subplot por controlador)
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    # 5 controladores + 1 célula vazia na grade 2×3
    fig.suptitle('Custo Acumulado por Modelo — última run', fontsize=13, fontweight='bold')
    labels_d = ['d=2', 'd=3', 'd=4', 'd=5']
    cores_d  = ['gray', 'blue', 'green', 'red']
    axs_flat = axes.flat
    for nome, _, _ in cfg:
        ax = next(axs_flat)
        for j, (lbl, cd) in enumerate(zip(labels_d, cores_d)):
            ax.semilogy(t, res[nome]['custos'][:, j], color=cd,
                        label=lbl, alpha=0.8, linewidth=1.3)
        ax.axvline(x=k_change, color='k', linestyle='--', linewidth=1)
        ax.set_title(nome, color=res[nome]['cor'], fontweight='bold', fontsize=11)
        ax.set_ylabel('Custo (log)');  ax.legend(fontsize=8);  ax.grid(True)
    next(axs_flat).set_visible(False)  # célula vazia
    plt.tight_layout()
    plt.show()

    # FIGURA 3 — Variância de b1 (Incerteza) por modelo
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle('Variância do Parâmetro b1 (Incerteza) por Modelo — última run',
                 fontsize=13, fontweight='bold')
    labels_m = ['Atraso 2', 'Atraso 3', 'Atraso 4', 'Atraso 5']
    cores_m  = ['blue', 'orange', 'green', 'red']
    axs_flat = axes.flat
    for nome, _, _ in cfg:
        ax = next(axs_flat)
        for i, (lbl, cm) in enumerate(zip(labels_m, cores_m)):
            ax.plot(t, res[nome]['incerteza'][:, i], color=cm,
                    label=lbl, linewidth=1.5)
        ax.axvline(x=k_change, color='k', linestyle='--', linewidth=1)
        ax.set_title(nome, color=res[nome]['cor'], fontweight='bold', fontsize=11)
        ax.set_yscale('log');  ax.set_ylabel('Variância')
        ax.legend(fontsize=8);  ax.grid(True, which='both', ls='-', alpha=0.4)
    next(axs_flat).set_visible(False)
    plt.tight_layout()
    plt.show()



    # TABELA DE DESEMPENHO MONTE CARLO
    # Calculada sobre as N_MC runs já executadas acima. -> Descarta os primeiros 40 passos (fase PRBS) para não distorcer as médias

    INICIO = 40  # passos descartados (fase de inicialização PRBS)

    print("\n" + "="*85)
    print(f"  TABELA DE DESEMPENHO — Monte Carlo  (N_MC={N_MC}, passos avaliados={N-INICIO})")
    print("="*85)
    print(f"  {'Controlador':<22} | {'V̄ (Perda Média)':<18} | {'σᵥ (Desvio Padrão)':<20} | {'p̄_b1 (Var. Média b1)'}")
    print("-"*85)

    for nome, _, _ in cfg:
        # Erro de rastreamento por run, descartando PRBS
        erros_mc = ((res[nome]['all_map'][:, INICIO:] - map_ref) ** 2).mean(axis=1)
        V_bar     = erros_mc.mean()               # perda média entre runs

        # Desvio padrão do erro de rastreamento (passo a passo, média entre runs)
        std_mc    = (res[nome]['all_map'][:, INICIO:] - map_ref).std(axis=1)
        sigma_v   = std_mc.mean()

        # Variância média de b1 ao final
        pb1_bar   = res[nome]['incerteza'][:, :].mean()
        print(f"  {nome:<22} | {V_bar:<18.4f} | {sigma_v:<20.4f} | {pb1_bar:.6f}")

    print("="*85)
    print("  V̄: menor = melhor rastreamento")
    print("  σᵥ: menor = mais robusto ao ruído")
    print("  p̄_b1: menor = RLS mais confiante nos parâmetros")
    print("="*85 + "\n")


In [7]:
# SLIDERS

w_ro     = widgets.FloatSlider(value=1.5, min=0.1, max=5.0,  step=0.1,   description='Peso (ro)')
w_lambda = widgets.FloatSlider(value=0.99, min=0.90, max=1.0, step=0.001, description='Lambda')
w_P0     = widgets.FloatSlider(value=150, min=100, max=180,   step=1,     description='P0 (mmHg)')
w_ref    = widgets.FloatSlider(value=100, min=50,  max=150,   step=1,     description='Ref (mmHg)')
w_sigma  = widgets.BoundedFloatText(value=1, min=0, max=5,               description='Ruído (σ)')

w_a1_ini = widgets.FloatText(value=-0.741, description='a1 (ini)')
w_b1_ini = widgets.FloatText(value=0.187,  description='b1 (ini)')
w_bm_ini = widgets.FloatText(value=0.075,  description='bm (ini)')
w_d_ini  = widgets.IntText(value=3,        description='d (ini)')
w_m_ini  = widgets.IntText(value=3,        description='m (ini)')

w_k_change = widgets.IntSlider(value=200, min=10, max=450, description='Instante (k)')
w_a1_new   = widgets.FloatText(value=-0.741, description='Novo a1')
w_b1_new   = widgets.FloatText(value=0.3,    description='Novo b1')
w_bm_new   = widgets.FloatText(value=0.075,  description='Novo bm')
w_d_new    = widgets.IntText(value=4,        description='Novo d')
w_m_new    = widgets.IntText(value=4,        description='Novo m')

tab1 = widgets.VBox([w_ro, w_lambda, w_P0, w_ref, w_sigma])
tab2 = widgets.VBox([w_a1_ini, w_b1_ini, w_bm_ini, w_d_ini, w_m_ini])
tab3 = widgets.VBox([w_k_change, w_a1_new, w_b1_new, w_bm_new, w_d_new, w_m_new])
ui   = widgets.Tab(children=[tab1, tab2, tab3])
ui.set_title(0, 'Controlador')
ui.set_title(1, 'Paciente Inicial')
ui.set_title(2, 'Mudança Parâmetros')

out = widgets.interactive_output(run_simulation, {
    'ro': w_ro, 'lambda_val': w_lambda, 'P0': w_P0, 'map_ref': w_ref, 'sigma': w_sigma,
    'a1_ini': w_a1_ini, 'b1_ini': w_b1_ini, 'bm_ini': w_bm_ini,
    'd_ini': w_d_ini, 'm_ini': w_m_ini,
    'k_change': w_k_change, 'a1_new': w_a1_new, 'b1_new': w_b1_new,
    'bm_new': w_bm_new, 'd_new': w_d_new, 'm_new': w_m_new,
})
display(ui, out)

Output()